# Explainable Credit Scoring Engine with SHAP-Based Decision Reasons

**Domain:** FinTech / Credit Risk / Explainable AI  
**Primary Evaluation Focus:** AUC-ROC, F1 Score, Interpretability  
**Dataset:** Home Credit-style application data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Credit scoring requires explainability as well as predictive performance.
- SHAP values can generate adverse-action style reason codes.
- Class imbalance is a core issue in default prediction.
- Regulated financial models require auditability, transparency and governance.
- Simple scorecard-style models may be preferable where compliance is more important than marginal accuracy.

## Business Recommendations

- Use explainability outputs to support fair and transparent credit decisions.
- Maintain separate model validation and governance documentation.
- Monitor approval rates and adverse impact across customer segments.
- Use calibrated probabilities for risk-based pricing and credit limits.
- Apply threshold tuning according to risk appetite.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **AUC-ROC, F1 Score, Interpretability**.

# Project 04 - Credit Scoring Engine: Explainable AI (FCA-Compliant)
**Domain:** FinTech / AI Engineering

FCA-compliant credit scoring with SHAP-based adverse action reasons for every decision.

**Dataset:** [Home Credit Default Risk on Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk/data) — use `application_train.csv`

In [ ]:
# Install: pip install scikit-learn shap imbalanced-learn pandas matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("Libraries loaded")

## 1. Load & Select Features

In [ ]:
df = pd.read_csv('application_train.csv')
print(f"Shape: {df.shape} | Default rate: {df['TARGET'].mean():.2%}")

features = ['TARGET','AMT_CREDIT','AMT_INCOME_TOTAL','AMT_ANNUITY',
            'DAYS_BIRTH','DAYS_EMPLOYED','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3',
            'REGION_RATING_CLIENT','CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY',
            'CNT_CHILDREN','AMT_GOODS_PRICE','DAYS_LAST_PHONE_CHANGE']

df2 = df[features].copy()
df2.fillna(df2.median(numeric_only=True), inplace=True)
df2['CODE_GENDER']    = (df2['CODE_GENDER']=='M').astype(int)
df2['FLAG_OWN_CAR']   = (df2['FLAG_OWN_CAR']=='Y').astype(int)
df2['FLAG_OWN_REALTY'] = (df2['FLAG_OWN_REALTY']=='Y').astype(int)
df2['AGE']            = -df2['DAYS_BIRTH'] / 365
df2['YEARS_EMPLOYED'] = -df2['DAYS_EMPLOYED'].clip(upper=0) / 365
df2['DEBT_TO_INCOME'] = df2['AMT_CREDIT'] / (df2['AMT_INCOME_TOTAL'] + 1)
df2['ANNUITY_TO_INCOME'] = df2['AMT_ANNUITY'] / (df2['AMT_INCOME_TOTAL'] + 1)
df2.drop(columns=['DAYS_BIRTH','DAYS_EMPLOYED','DAYS_LAST_PHONE_CHANGE'], inplace=True)

X = df2.drop('TARGET', axis=1)
y = df2['TARGET']
print(f"Features: {list(X.columns)}")

## 2. SMOTE + Train Logistic Regression Scorecard

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_tr, y_train)
print(f"After SMOTE: {dict(zip(*np.unique(y_res, return_counts=True)))}")

lr = LogisticRegression(max_iter=1000, C=0.1, class_weight='balanced', random_state=42)
lr.fit(X_res, y_res)
prob = lr.predict_proba(X_te)[:,1]
pred = lr.predict(X_te)

auc = roc_auc_score(y_test, prob)
gini = 2*auc - 1
print(f"AUC-ROC: {auc:.4f} | Gini: {gini:.4f}")
print(classification_report(y_test, pred, target_names=['Good','Default']))

## 3. SHAP Explainability

In [ ]:
explainer = shap.LinearExplainer(lr, X_res)
shap_values = explainer.shap_values(X_te[:500])

shap.summary_plot(shap_values, X_test.iloc[:500], feature_names=X.columns.tolist())

## 4. Per-Customer Adverse Action Reasons (FCA-Compliant)

In [ ]:
def adverse_action_report(idx):
    sv = shap_values[idx]
    prob_default = prob[idx]
    decision = 'DECLINED' if prob_default > 0.5 else 'APPROVED'
    risk_factors = sorted(zip(X.columns, sv), key=lambda x: x[1], reverse=True)

    print(f"=== Application {idx} ===")
    print(f"Decision: {decision} | Default probability: {prob_default:.1%}")
    print("\nTop 3 Adverse Action Reasons (FCA Compliant):")
    for i, (feat, val) in enumerate(risk_factors[:3], 1):
        print(f"  {i}. {feat}: {'increases' if val>0 else 'decreases'} risk (SHAP: {val:+.4f})")
    print()

adverse_action_report(0)
adverse_action_report(5)
adverse_action_report(10)

## 5. ROC Curve & Scorecard Coefficients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, prob)
axes[0].plot(fpr, tpr, color='#C9A96E', lw=2, label=f'AUC = {auc:.4f}')
axes[0].plot([0,1],[0,1],'--',color='gray')
axes[0].set_title('ROC Curve — Credit Scorecard', fontweight='bold')
axes[0].legend()

coef = pd.Series(lr.coef_[0], index=X.columns).sort_values()
coef.plot(kind='barh', ax=axes[1], color=['#C9A96E' if v>0 else '#0F1C2E' for v in coef])
axes[1].axvline(0, color='black', lw=0.5)
axes[1].set_title('Scorecard Coefficients\n(positive = higher default risk)', fontweight='bold')

plt.tight_layout()
plt.show()

# Final Business Interpretation

## Key Findings

- Credit scoring requires explainability as well as predictive performance.
- SHAP values can generate adverse-action style reason codes.
- Class imbalance is a core issue in default prediction.
- Regulated financial models require auditability, transparency and governance.
- Simple scorecard-style models may be preferable where compliance is more important than marginal accuracy.

## Recommended Next Steps

- Use explainability outputs to support fair and transparent credit decisions.
- Maintain separate model validation and governance documentation.
- Monitor approval rates and adverse impact across customer segments.
- Use calibrated probabilities for risk-based pricing and credit limits.
- Apply threshold tuning according to risk appetite.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Built an explainable credit scoring workflow using machine learning and SHAP-based reason codes, focusing on model transparency, risk governance and financial services compliance."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining